# feeding / exchanger 동작 판별 모델

두 모델이 답하는 질문이 다르다.

| | 질문 | 근거 신호 | 확답 가능? |
|---|---|---|---|
| **feeding** | ① 명령 후 실제로 들어왔나 ② 평소보다 약한가 ③ 늦게 열리나 | **수위** (직접 계측) | **예** |
| **exchanger** | 이 동작이 평소와 다른가 | 탱크 온도 (간접) | **아니오** — 이상 후보만 제시 |

**왜 차이가 나나.** feeding 은 결과가 수위에 직접 찍힌다. 명령이 나갔는데 수위가
안 오르면 그건 논쟁의 여지가 없다. exchanger 는 그런 계측이 없다 —
`Scale_Out___TT_EX_*` 라는 전용 온도센서 태그가 있지만 **값이 전부 0 인 미사용
채널**이었다. 그래서 탱크 온도로 간접 추정할 수밖에 없고, "안 일어났다"를
단정할 수 없다.

## 두 모델의 공통 설계 원칙

**1. 이벤트가 관측 단위다.** 5초 격자로 보면 40만 행이지만 인접 샘플은 거의 같은
값이라 독립 정보가 아니다. 앞서 표본 단위로 검출력을 계산했다가 실제보다 크게
과대평가한 적이 있다. 이벤트 하나 = 관측 하나로 센다.

**2. 조건을 통제한다.** exchanger 는 "온도가 높을 때 켜진다"는 선택 편향이 있고,
이벤트마다 시작 온도·지속시간·heat 상태가 다르다. 이걸 회귀로 빼내지 않으면
조건 차이가 고장 신호를 덮는다. 실측으로 잔차 σ 가 11~35% 줄었다.

**3. 기준선은 직전 N개 이벤트로 잡는다(이동 기준선).** 고정 기준선을 쓰면 4.7일에
걸친 정상 공정 변동까지 이상으로 쌓여 멀쩡한 탱크가 전부 "고장"으로 나온다.
릴레이 고장은 서서히가 아니라 **한순간에 끊기는 계단형 변화**라 직전 이벤트와
비교해도 놓치지 않는다.

**4. 중앙값/MAD 로 z 를 낸다.** 고장 이벤트 몇 개가 기준선 자체를 끌어당기는 걸
막는다. z 는 ±4 로 자른다 — 자르지 않으면 튀는 이벤트 하나로 CUSUM 이 폭발한다
(실측: P1 feeding CUSUM 최대값 238 → 23).

**5. 임계값은 실측으로 정한다.** 교과서 기본값(5)을 쓰면 안 된다. 정상 데이터의
CUSUM 최대값이 탱크마다 4~57 로 제각각이라 5 로 두면 전부 오경보다.

## 0. 설정

In [ ]:
import glob
import os
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, os.getcwd())
from Tank_Temp_Factor_Analysis import TANKS, WAGON_TAG

import matplotlib
import matplotlib.pyplot as plt
# ↑ 위 import 가 백엔드를 Agg 로 바꾸므로 inline 은 그 뒤에
%matplotlib inline
matplotlib.rcParams["font.family"] = "Malgun Gothic"
matplotlib.rcParams["axes.unicode_minus"] = False

SAVE_DIR = "extracted_csv"

EXCLUDE_HOURS = (0, 7)
RUNNING_WINDOW_MIN = 30      # wagon 이 이 시간 안에 안 움직이면 라인 정지로 본다
MIN_EVENT_SEC = 5            # I2/I3 의 feeding 펄스가 중앙값 7.3초라 10초로 잡으면 안 된다
TAIL_SEC = 60                # 이벤트 종료 후 반응을 더 볼 시간

BASELINE_WINDOW = 30         # 이동 기준선에 쓸 직전 이벤트 수
Z_CLIP = 4.0
Z_WARN = 3.0                 # 개별 이벤트 "이상" 판정
CUSUM_K = 0.5
THRESH_MARGIN = 1.3          # 정상 CUSUM 최대값 × 이 값 = 확정 임계값

#: feeding "발생" 판정 기준 — 그 탱크의 평소 유입량 대비 비율.
#: 수위 센서 잡음(σ=0.04~0.08)에 비해 평소 유입량이 9~10 이라 한참 위이고,
#: 정상 변동(변동계수 0.08~0.30)에 비해서는 한참 아래라 오탐/미탐 둘 다 피한다.
OCCURRED_FRAC = 0.25

## 1. 데이터 로드 — 정지/알람 구간 제외

In [ ]:
_CACHE = {}


def load(tank_key):
    if tank_key in _CACHE:
        return _CACHE[tank_key]

    cfg = TANKS[tank_key]
    path = sorted(glob.glob(os.path.join(SAVE_DIR, f"{tank_key}_Temp_vs_Factors_*.csv")))[-1]
    use = ["Time", "source", WAGON_TAG, cfg["temp"], "level_f",
           cfg["exchanger"], cfg["feeding"], cfg["heat"]]
    raw = pd.read_csv(path, index_col="Time", parse_dates=True, usecols=use).sort_index()

    d = pd.DataFrame({
        "T": raw[cfg["temp"]], "level": raw["level_f"],
        "exch": raw[cfg["exchanger"]], "feed": raw[cfg["feeding"]],
        "heat": raw[cfg["heat"]], "wagon": raw[WAGON_TAG],
        "diag": (raw["source"] == "DIAGNOSTIC").astype(float),
    })
    w = d["wagon"].resample("1min").last().ffill()
    run = (w.diff().abs() > 0).rolling(RUNNING_WINDOW_MIN, min_periods=1).max()
    d["running"] = run.reindex(d.index, method="ffill").fillna(0)
    h = d.index.hour
    d["ok"] = (((h < EXCLUDE_HOURS[0]) | (h >= EXCLUDE_HOURS[1]))
               & (d["diag"] == 0) & (d["running"] == 1))

    _CACHE[tank_key] = d
    print(f"[{tank_key}] {len(d):,}행 | 유효 {int(d['ok'].sum()):,}행")
    return d


def events(d, col):
    s = (d[col] == 1) & d["ok"]
    grp = (s & ~s.shift(1, fill_value=False)).cumsum()[s]
    out = []
    for _, idx in d.index[s].to_series().groupby(grp):
        if (idx.iloc[-1] - idx.iloc[0]).total_seconds() >= MIN_EVENT_SEC:
            out.append((idx.iloc[0], idx.iloc[-1]))
    return out


for tk in TANKS:
    load(tk)

## 2. 공통 통계 도구

In [ ]:
def _mad_scale(base):
    med = np.median(base)
    mad = np.median(np.abs(base - med))
    scale = 1.4826 * mad
    if scale < 1e-9:
        scale = np.std(base) if np.std(base) > 1e-9 else 1.0
    return med, scale


def rolling_z(values, window=BASELINE_WINDOW, clip=Z_CLIP):
    """직전 window 개를 기준선으로 한 로버스트 z-score (중앙값/MAD, ±clip 절단)."""
    z = np.full(len(values), np.nan)
    for i in range(window, len(values)):
        med, scale = _mad_scale(values[i - window:i])
        z[i] = (values[i] - med) / scale
    return np.clip(z, -clip, clip)


def cusum_down(z, k=CUSUM_K):
    """반응이 기준보다 계속 작으면 쌓인다. 부호 정규화 후에 넣을 것."""
    s, out = 0.0, []
    for v in z:
        s = 0.0 if np.isnan(v) else max(0.0, s - v - k)
        out.append(s)
    return np.array(out)


def grade(z, cs, thr):
    """개별 이벤트 등급 — z 는 단발 이상, CUSUM 은 지속 이상을 본다."""
    if not np.isnan(cs) and cs >= thr:
        return "확정"
    if (not np.isnan(cs) and cs >= thr * 0.6) or (not np.isnan(z) and z <= -Z_WARN):
        return "주의"
    return "정상"

## 3. feeding 모델

이벤트마다 네 가지를 잰다.

| 지표 | 계산 | 무엇을 잡나 |
|---|---|---|
| **유입량** | 수위(끝+60초) − 수위(시작−5초) | 아예 안 들어왔는지 |
| **유량** | 유입량 ÷ 지속시간 | 평소보다 약한지 (막힘·마모) |
| **지연** | 신호 ON 부터 수위가 잡음 3σ 를 넘을 때까지 | 밸브가 늦게 열리는지 (고착 전조) |
| **발생여부** | 유입량 ≥ 평소 중앙값 × 0.25 | 즉시 판정용 게이트 |

`TAIL_SEC` 만큼 더 보는 이유: 수위 반응에 지연이 있다 (I3 기준 +25초).

In [ ]:
def feeding_events(tank_key):
    d = load(tank_key)
    lv = d["level"]
    noise = lv.diff().abs().median() * 1.4826 + 1e-9      # 수위 센서 잡음 규모

    rows = []
    for t0, t1 in events(d, "feed"):
        pre = lv.asof(t0 - pd.Timedelta(seconds=5))
        post = lv.asof(t1 + pd.Timedelta(seconds=TAIL_SEC))
        dur = (t1 - t0).total_seconds()
        if pd.isna(pre) or pd.isna(post) or dur <= 0:
            continue

        # 지연: 신호 ON 후 수위가 잡음 3배를 넘어선 첫 시점
        seg = lv.loc[t0: t1 + pd.Timedelta(seconds=TAIL_SEC)]
        over = seg[seg > pre + 3 * noise]
        delay = (over.index[0] - t0).total_seconds() if len(over) else np.nan

        rows.append({"start": t0, "지속초": round(dur, 1),
                     "유입량": round(post - pre, 3),
                     "유량": (post - pre) / dur,
                     "지연초": round(delay, 1) if delay == delay else np.nan})
    return pd.DataFrame(rows)


def feeding_model(tank_key):
    e = feeding_events(tank_key)
    if len(e) < BASELINE_WINDOW + 10:
        return e, {"탱크": tank_key, "이벤트": len(e), "판정": "표본부족"}

    med_in = e["유입량"].median()
    e["발생"] = e["유입량"] >= med_in * OCCURRED_FRAC

    e["z_유량"] = rolling_z(e["유량"].values.astype(float))
    e["z_지연"] = -rolling_z(e["지연초"].fillna(e["지연초"].median()).values.astype(float))
    e["cusum"] = cusum_down(e["z_유량"].values)

    cs = e["cusum"].dropna()
    thr = max(float(cs.max()) * THRESH_MARGIN, 5.0) if len(cs) else 5.0
    e["판정"] = [grade(z, cu, thr) for z, cu in zip(e["z_유량"], e["cusum"])]

    info = {"탱크": tank_key, "이벤트": len(e),
            "미발생": int((~e["발생"]).sum()),
            "유량중앙값": round(e["유량"].median(), 4),
            "지연중앙값": round(e["지연초"].median(), 1),
            "임계값": round(thr, 1),
            "현재CUSUM": round(float(e["cusum"].iloc[-1]), 2),
            "판정": grade(e["z_유량"].iloc[-1], e["cusum"].iloc[-1], thr)}
    return e, info


FEED, rows = {}, []
for tk in TANKS:
    FEED[tk], info = feeding_model(tk)
    rows.append(info)
pd.DataFrame(rows)

## 4. exchanger 모델

**이벤트 하나 = 관측 하나**로 두고, 조건을 회귀로 통제한 뒤 남은 잔차를 본다.

조건으로 넣는 것: **시작 온도**(높을수록 더 떨어짐 — 선택 편향의 원인),
**지속시간**, **heat 비율**, **직전 온도 기울기**(이미 내려가던 중이었는지).

이 네 가지를 빼내면 잔차 σ 가 11~35% 줄어든다. 남은 잔차가 "조건이 같았는데도
평소와 달랐던 정도"다.

⚠️ **이 모델은 "exchange 가 일어났다/안 났다"를 확정하지 못한다.** 전용 계측이
없어서 온도로 간접 추정하는 것이고, 실측 효과크기가 P3·I1 에서는 0.4 수준이라
그 두 탱크는 사실상 판별 불가다. **이상 후보를 사람에게 넘기는 용도**로만 써야 한다.

In [ ]:
EXCH_FEATURES = ["시작온도", "지속초", "heat비율", "직전기울기"]


def exch_events(tank_key):
    d = load(tank_key)
    rows = []
    for t0, t1 in events(d, "exch"):
        seg = d.loc[t0: t1 + pd.Timedelta(seconds=TAIL_SEC), "T"].dropna()
        pre = d.loc[t0 - pd.Timedelta(seconds=180): t0, "T"].dropna()
        if len(seg) < 5 or len(pre) < 5:
            continue
        xp = (pre.index - pre.index[0]).total_seconds().values
        rows.append({"start": t0,
                     "온도변화": seg.iloc[-1] - seg.iloc[0],
                     "시작온도": seg.iloc[0],
                     "지속초": (t1 - t0).total_seconds(),
                     "heat비율": d.loc[t0:t1, "heat"].mean(),
                     "직전기울기": np.polyfit(xp, pre.values, 1)[0] * 60})
    return pd.DataFrame(rows)


def exch_model(tank_key):
    e = exch_events(tank_key)
    if len(e) < BASELINE_WINDOW + 30:
        return e, {"탱크": tank_key, "이벤트": len(e), "판정": "표본부족"}

    A = np.column_stack([np.ones(len(e))] + [e[f].values for f in EXCH_FEATURES])
    y = e["온도변화"].values
    beta = np.linalg.pinv(A.T @ A) @ A.T @ y
    e["기대변화"] = A @ beta
    e["잔차"] = y - e["기대변화"]

    # 고장 = 효과가 0 쪽으로 = 잔차가 기대효과 반대 방향으로. 부호 정규화 후 CUSUM
    sign = np.sign(np.median(y))
    e["z"] = rolling_z((e["잔차"].values) * sign)
    e["cusum"] = cusum_down(e["z"].values)

    cs = e["cusum"].dropna()
    thr = max(float(cs.max()) * THRESH_MARGIN, 5.0) if len(cs) else 5.0
    e["판정"] = [grade(z, cu, thr) for z, cu in zip(e["z"], e["cusum"])]

    # **"검출력" 같은 숫자를 여기서 만들면 안 된다.** exchanger 의 진짜 인과 효과를
    # 이 데이터로는 추정할 수 없다 (모든 이벤트가 ON 이라 OFF 반사실이 없다).
    # 잔차 산포만 정직하게 보고하고, 민감도는 아래 5번에서 "얼마만큼 달라지면
    # 알아채는가" 형태로 제시한다.
    info = {"탱크": tank_key, "이벤트": len(e),
            "단순σ": round(float(np.std(y)), 2), "조건통제σ": round(float(e["잔차"].std()), 2),
            "설명력": f"{(1 - e['잔차'].std() / np.std(y)) * 100:.0f}%",
            "임계값": round(thr, 1),
            "현재CUSUM": round(float(e["cusum"].iloc[-1]), 2),
            "판정": grade(e["z"].iloc[-1], e["cusum"].iloc[-1], thr)}
    return e, info


EXCH, rows = {}, []
for tk in TANKS:
    EXCH[tk], info = exch_model(tk)
    rows.append(info)
pd.DataFrame(rows)

## 5. 성능 검증 — 얼마나 달라지면 알아채는가

실제 고장 사례가 데이터에 없으므로 가짜로 주입해서 잰다. **두 모델의 주입
방식이 다르다.**

- **feeding**: 유량을 **0 으로** 만든다. "명령은 갔는데 안 들어옴" 이 실제로
  일어날 수 있는 고장 상태이고, 수위로 직접 관측되므로 이게 그대로 시뮬레이션이 된다.
- **exchanger**: 잔차를 0 으로 만드는 건 **시뮬레이션이 아니다** — 잔차는 회귀로
  이미 평균이 0 이라 아무 일도 안 일어난다. 그리고 exchanger 의 진짜 효과 크기를
  이 데이터로는 알 수 없다(모든 이벤트가 ON 이라 OFF 반사실이 없다). 그래서
  **"반응이 잔차 σ 의 1배/2배만큼 약해지면 몇 번 만에 알아채는가"** 형태로
  민감도를 제시한다. 절대적인 고장 검출 능력이 아니라 **상대적 민감도**다.

In [ ]:
def _detect_after(v, sign, thr, n):
    cs = cusum_down(rolling_z(v * sign))[-n:]
    return int(np.argmax(cs >= thr)) + 1 if (cs >= thr).any() else None


def injection(e, metric, mode, n=20):
    """mode='zero' 는 반응을 0 으로, mode=(배수) 는 잔차 σ 의 그 배수만큼 약화시킨다."""
    if len(e) < BASELINE_WINDOW + 30:
        return None
    v = e[metric].values.astype(float)
    sign = np.sign(np.median(v)) or 1.0
    thr = max(float(cusum_down(rolling_z(v * sign)).max()) * THRESH_MARGIN, 5.0)

    vi = v.copy()
    if mode == "zero":
        vi[-n:] = 0.0
    else:
        vi[-n:] = vi[-n:] - sign * mode * np.std(v)
    return _detect_after(vi, sign, thr, n)


rows = []
for tk in TANKS:
    rows.append({
        "탱크": tk,
        "feeding_유량0": injection(FEED[tk], "유량", "zero"),
        "exch_1σ약화": injection(EXCH[tk], "잔차", 1.0),
        "exch_2σ약화": injection(EXCH[tk], "잔차", 2.0),
        "exch_3σ약화": injection(EXCH[tk], "잔차", 3.0),
    })
print("몇 번째 이벤트에서 확정 알람이 뜨는가 (None = 20개 안에 못 잡음)")
pd.DataFrame(rows)

## 6. 운영 — 어떤 판단에 어떻게 쓰나

| 모델 출력 | 무엇을 말하나 | 확실성 | 대응 |
|---|---|---|---|
| feeding **미발생** | 명령은 갔는데 수위가 안 올랐다 | **높음** (직접 계측) | **즉시 현장 확인** — 릴레이/솔레노이드/배관 |
| feeding **z_유량 ≤ −3** | 이번에 평소보다 약하게 들어왔다 | 중간 | 관찰. 반복되면 필터 막힘·밸브 마모 의심 |
| feeding **CUSUM ≥ 임계** | 여러 이벤트에 걸쳐 계속 약하다 | **높음** | **정비 일정 잡기** — 서서히 진행되는 열화 |
| feeding **지연 증가** | 밸브가 늦게 열린다 | 중간 | 고착 전조. 지연 추세를 같이 보기 |
| exchanger **z ≤ −3** | 조건이 같았는데 이번엔 달랐다 | **낮음** | 검토 목록에만 올림. 단독으로 정비 근거 못 됨 |
| exchanger **CUSUM ≥ 임계** | 지속적으로 다르다 | 중간 | 다른 근거와 함께 점검 요청 |

### 판정 3단계를 둔 이유

"정상 / 주의 / 확정" 으로 나눴다. 이진 판정은 임계값 근처에서 깜빡이고,
오경보 한 번이면 현장에서 알람 자체를 무시하게 된다. **주의** 는 "지금 당장
사람을 부를 정도는 아니지만 다음 이벤트를 눈여겨보라" 는 뜻이다.

### 탱크별 신뢰도 차이를 반드시 감안할 것

같은 모델이라도 탱크마다 민감도가 다르다. 위 5번 표에서 `exch_2σ약화` 가
None 인 탱크는 어지간한 변화로는 알람이 안 뜬다는 뜻이다. feeding 은 P4(이벤트 6개)와
I2(35개)가 표본 부족이라 CUSUM 기준선을 아직 못 만든다 — 데이터가 더 쌓여야 한다.

### 이 모델이 못 하는 것

- **exchanger 의 "안 일어났다" 단정** — 전용 계측이 없다
- **원인 구분** — 릴레이 고장인지, 솔레노이드 고장인지, 배관 막힘인지 구분 못 한다.
  "명령 대비 결과가 안 나왔다" 까지만 말한다
- **급격한 공정 변경 대응** — 레시피가 바뀌어 정상 유량이 달라지면 이동 기준선이
  따라잡을 때까지(약 30 이벤트) 오경보가 날 수 있다

## 7. 탱크별 상세 보기

In [ ]:
TANK = "I3"

fig, axes = plt.subplots(3, 2, figsize=(15, 10))
f, x = FEED[TANK], EXCH[TANK]

axes[0][0].plot(f["start"], f["유량"], ".", ms=4, color="#1f77b4")
axes[0][0].axhline(f["유량"].median(), color="#d62728", ls="--", lw=1, label="중앙값")
axes[0][0].set_title(f"{TANK} feeding — 유량 (수위/초)"); axes[0][0].legend(fontsize=8)

axes[1][0].plot(f["start"], f["지연초"], ".", ms=4, color="#9467bd")
axes[1][0].set_title("feeding — 반응 지연 (초)")

axes[2][0].plot(f["start"], f["cusum"], color="#2ca02c", lw=1.2)
cs = f["cusum"].dropna(); thr = max(cs.max() * THRESH_MARGIN, 5.0)
axes[2][0].axhline(thr * 0.6, color="orange", ls="--", lw=1, label="주의")
axes[2][0].axhline(thr, color="red", ls="--", lw=1, label="확정")
axes[2][0].set_title("feeding — CUSUM"); axes[2][0].legend(fontsize=8)

axes[0][1].scatter(x["기대변화"], x["온도변화"], s=6, alpha=0.4, color="#1f77b4")
lim = [x["온도변화"].min(), x["온도변화"].max()]
axes[0][1].plot(lim, lim, color="#d62728", ls="--", lw=1)
axes[0][1].set_xlabel("조건으로 예측한 온도변화"); axes[0][1].set_ylabel("실제")
axes[0][1].set_title(f"{TANK} exchanger — 조건 통제 적합")

axes[1][1].plot(x["start"], x["잔차"], ".", ms=4, color="#ff7f0e")
axes[1][1].axhline(0, color="#888888", lw=0.8)
axes[1][1].set_title("exchanger — 잔차")

axes[2][1].plot(x["start"], x["cusum"], color="#2ca02c", lw=1.2)
cs2 = x["cusum"].dropna(); thr2 = max(cs2.max() * THRESH_MARGIN, 5.0)
axes[2][1].axhline(thr2 * 0.6, color="orange", ls="--", lw=1, label="주의")
axes[2][1].axhline(thr2, color="red", ls="--", lw=1, label="확정")
axes[2][1].set_title("exchanger — CUSUM"); axes[2][1].legend(fontsize=8)

for a in axes.ravel():
    a.grid(alpha=0.3)
fig.autofmt_xdate()
fig.tight_layout()

## 8. 최근 이벤트 목록 (현장 확인용)

In [ ]:
LIST_TANK = "I3"
N = 15

print(f"=== {LIST_TANK} feeding 최근 {N}건 ===")
cols = ["start", "지속초", "유입량", "유량", "지연초", "발생", "z_유량", "cusum", "판정"]
display(FEED[LIST_TANK][cols].tail(N).round(3))

print(f"=== {LIST_TANK} exchanger 최근 {N}건 ===")
cols2 = ["start", "지속초", "시작온도", "온도변화", "기대변화", "잔차", "z", "cusum", "판정"]
display(EXCH[LIST_TANK][cols2].tail(N).round(2))